In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

In [4]:
def attention(query, key, value, mask=None, dropout=None):
    ''' 
    缩放点积注意力
    query: (bs, num_heads, seq_len, d_k)
    key: (bs, num_heads, seq_len, d_k)
    value: (bs, num_heads, seq_len, d_v)
    对于 torch.matmul 来说，只要张量维度大于等于 2，
    它永远只会把最后两个维度当成真正的“矩阵”去相乘，
    而把前面所有的维度都统统视为“Batch（批次）
    '''
    d_k = query.size(-1) 
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        pass

    p_attn = F.softmax(scores, dim=-1)

    if dropout is not None:
        p_attn = nn.Dropout(dropout)(p_attn)

    return torch.matmul(p_attn, value), p_attn
    

In [5]:
class MutilHeadedAttention(nn.Module):
    def __init__(self, num_heads, d_model, dropout=0.1):
        super().__init__()
        self.p_attn = None
        self.num_heads = num_heads
        self.d_model = d_model

        self.d_k = d_model // num_heads
        self.dropout = dropout
        # 从效率上，QKV不能真的用8个linear，所以我们把8个linear合并，也就是num_heads个d_k = d_model
        self.linears = nn.ModuleList(
            [nn.Linear(d_model, d_model), # query 的 Linear
            nn.Linear(d_model, d_model), # key 的 Linear
            nn.Linear(d_model, d_model), # value 的 Linear
            nn.Linear(d_model, d_model)]
        )

    
    def forward(self, query, key, value, mask=None):
        ''' 
        query --> (batch_size, query_len, d_model)
        '''
        batch_size = query.size(0)
        projected_query = self.linears[0](query) # (batch_size, query_len, d_model)
        projected_key = self.linears[1](key) # (batch_size, key_len, d_model)
        projected_value = self.linears[2](value) # (batch_size, value_len, d_model)

        # 多头切分
        viewed_query = projected_query.reshape(batch_size, -1, self.num_heads, self.d_k) # (batch_size, query_len, num_heads, d_k)
        viewed_key = projected_key.reshape(batch_size, -1, self.num_heads, self.d_k)
        viewed_value = projected_value.reshape(batch_size, -1, self.num_heads, self.d_k)

        # 调整维度，使得每个头独立
        transdim_query = viewed_query.transpose(1, 2) # (batch_size, num_heads, query_len, d_k)
        transdim_key = viewed_key.transpose(1, 2) # (batch_size, num_heads, key_len, d_k)
        transdim_value = viewed_value.transpose(1, 2) # (batch_size, num_heads, value_len, d_k)

        # 计算att
        att, self.p_attn = attention(transdim_query, transdim_key, transdim_value, mask=mask, dropout=self.dropout)

        # 多头重组，分开的多个头再合并回去，提高效率
        transdim_att = att.transpose(1, 2) # (batch_size, query_len, num_heads, d_k)
        transdim_att = transdim_att.contiguous()

        merge_att = transdim_att.reshape(batch_size, -1, self.d_model) # (batch_size, query_len, d_model)
        projected_att = self.linears[3](merge_att) # (batch_size, query_len, d_model)

        return projected_att


In [4]:
def test_mutil_head_att():
    bs = 2
    seq_len = 1024
    d_model = 512
    num_heads = 8
    dropout = 0.1

    mha = MutilHeadedAttention(num_heads, d_model, dropout)
    test_tensor = torch.randn(bs, seq_len, d_model)

    output = mha(test_tensor, test_tensor, test_tensor)
    print(output.shape)

test_mutil_head_att()

torch.Size([2, 1024, 512])


In [ ]:
class SubLayerConnection(nn.Module):
    ''' 
    给定a层和b层，自动构建 a + dropout(b(norm(a)))
    '''
    def __init__(self, size, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))
    

class EncoderLayer(nn.Module):
    def __init__(self, size, self_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = nn.ModuleList(
            SubLayerConnection(size, dropout),
            SubLayerConnection(size, dropout)
        )

    def forward(self, x, mask):
        # 这里一定得是一个函数
        def self_attention_forward(norm_x):
            return self.self_attn(
                query=norm_x,
                key=norm_x,
                value=norm_x,
                mask=mask
            )
        x = self.sublayer[0](x, self_attention_forward)
        x = self.sublayer[1](x, self.feed_forward)
        return x
    

In [7]:
class Encoder(nn.Module):
    def __init__(self, layer):
        super().__init__()
        self.layers = nn.ModuleList(
            layer, # 6层 多头注意力
            layer,
            layer,
            layer,
            layer,
            layer,
        )
        self.norm = nn.LayerNorm(layer.size)
    
    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
            
        return self.norm(x)

In [8]:
class Encoder(nn.Module):
    def __init__(self, d_model, self_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.norm = nn.LayerNorm(d_model)
        # self.encoder_norm = nn.LayerNorm()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # EncoderLayer
        # sublayer 0: self-attention
        x = x + self.dropout(self.self_attn(
                query=self.norm(x),
                key=self.norm(x),
                value=self.norm(x),
                mask=mask
            )[0])
        
        # sublayer 1: feed-forward
        x = x + self.dropout(self.feed_forward(self.norm(x)))

        # EncoderLayer
        # sublayer 0: self-attention
        x = x + self.dropout(self.self_attn(
                query=self.norm(x),
                key=self.norm(x),
                value=self.norm(x),
                mask=mask
            )[0])
        
        # sublayer 1: feed-forward
        x = x + self.dropout(self.feed_forward(self.norm(x)))

        # EncoderLayer
        # sublayer 0: self-attention
        x = x + self.dropout(self.self_attn(
                query=self.norm(x),
                key=self.norm(x),
                value=self.norm(x),
                mask=mask
            )[0])
        
        # sublayer 1: feed-forward
        x = x + self.dropout(self.feed_forward(self.norm(x)))

        # EncoderLayer
        # sublayer 0: self-attention
        x = x + self.dropout(self.self_attn(
                query=self.norm(x),
                key=self.norm(x),
                value=self.norm(x),
                mask=mask
            )[0])
        
        # sublayer 1: feed-forward
        x = x + self.dropout(self.feed_forward(self.norm(x)))

        # EncoderLayer
        # sublayer 0: self-attention
        x = x + self.dropout(self.self_attn(
                query=self.norm(x),
                key=self.norm(x),
                value=self.norm(x),
                mask=mask
            )[0])
        
        # sublayer 1: feed-forward
        x = x + self.dropout(self.feed_forward(self.norm(x)))

        # EncoderLayer
        # sublayer 0: self-attention
        x = x + self.dropout(self.self_attn(
                query=self.norm(x),
                key=self.norm(x),
                value=self.norm(x),
                mask=mask
            )[0])
        
        # sublayer 1: feed-forward
        x = x + self.dropout(self.feed_forward(self.norm(x)))

        x = self.norm(x)
        return x

        
        

        
        
        
        

In [7]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU(replace=True)
    def forward(self, x):
        x = self.w_1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.w_2(x)
        return x


In [ ]:
d_model = 512        # 隐藏层维度
d_ff = 2048          # FFN 膨胀维度
num_heads = 8
dropout = 0.1

attn = MutilHeadedAttention(num_heads=num_heads, d_model=d_model, dropout=dropout)
ff = PositionWiseFeedForward(d_model=d_model, d_ff=d_ff, dropout=dropout)

encoder = Encoder(d_model=d_model, self_attn=attn, feed_forward=ff, dropout=dropout)